# Advanced Training Strategies for Pneumonia Detection in X-Ray Images
---

Prof. Dr.-Ing. Antje Muntzinger, Hochschule für Technik Stuttgart

antje.muntzinger@hft-stuttgart.de

---

This notebook demonstrates how to train a convolutional neural network on the
[Chest X-Ray Pneumonia Dataset](https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia).

The images in this database are grayscale x-ray images; some example images are pictured below.

<img src='images/pneunomia.png' width=50% />


In [ ]:
import torch
from torch import nn, optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np

# set random seeds for reproducibility
import random
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    
set_seed(42)

# check if CUDA is available
train_on_gpu = torch.cuda.is_available()

if not train_on_gpu:
    print('CUDA is not available.  Training on CPU ...')
    device = torch.device("cpu")
else:
    print('CUDA is available!  Training on GPU ...')
    device = torch.device("cuda")

## Load and Augment the Data

First, we download the data from https://bwsyncandshare.kit.edu/s/WodFpTgkRiaMfgd, save it and extract the zip file in a subfolder "data" of the working directory. 

In [ ]:
from pathlib import Path
import requests
import zipfile

# Download and extract the dataset
url = "https://bwsyncandshare.kit.edu/s/WodFpTgkRiaMfgd/download"
data_dir = Path("data")
data_dir.mkdir(exist_ok=True)

zip_path = data_dir / "chest_xray.zip"

# Download
if not zip_path.exists():
    response = requests.get(url, timeout=120)
    response.raise_for_status()
    zip_path.write_bytes(response.content)

# Extract
with zipfile.ZipFile(zip_path) as z:
    z.extractall(data_dir)

# Remove ZIP file
zip_path.unlink()

print("Dataset ready in:", data_dir.resolve())

We load in the training, validation and test data, then create DataLoaders for each of these sets of data.

In [ ]:
IMG_SIZE = 64  # or 128

# Data transforms and loaders
train_transforms = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

test_transforms = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

def load_train_val(train_transform, val_transform, val_fraction=0.2):
    # Both datasets initially contain all images from the training folder.
    train = datasets.ImageFolder(
        './data/chest_xray/train', transform=train_transform
    )
    val = datasets.ImageFolder(
        './data/chest_xray/train', transform=val_transform
    )

    # Reusing the seed gives every model the same stratified split.
    rng = np.random.default_rng(42)
    targets = np.array(train.targets)
    train_indices, val_indices = [], []

    for label in np.unique(targets):
        indices = rng.permutation(np.flatnonzero(targets == label))
        n_val = int(round(val_fraction * len(indices)))
        val_indices.extend(indices[:n_val])
        train_indices.extend(indices[n_val:])

    # Keep ImageFolder attributes compatible with the remaining notebook.
    for dataset, indices in [(train, train_indices), (val, val_indices)]:
        dataset.samples = [dataset.samples[i] for i in indices]
        dataset.imgs = dataset.samples
        dataset.targets = [label for _, label in dataset.samples]

    return train, val

# Split the original training folder into 80% training and 20% validation.
train_data, val_data = load_train_val(
    train_transforms, test_transforms, val_fraction=0.2
)

# Load the separate test set.
test_data = datasets.ImageFolder(
    './data/chest_xray/test', transform=test_transforms
)

batch_size = 32
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_data, batch_size=batch_size, shuffle=False)
test_loader  = DataLoader(test_data, batch_size=batch_size, shuffle=False)

classes = train_data.classes
print("Classes:", classes)
print(f"Training:   {len(train_data)} images")
print(f"Validation: {len(val_data)} images")
print(f"Test:       {len(test_data)} images")

### Visualize a Batch of Training Data

In [ ]:
# helper function to un-normalize and display an image
def imshow(img):
    img = img / 2 + 0.5  # unnormalize
    img = np.squeeze(img) # remove single-dimensional entries from the shape
    plt.imshow(img, cmap='gray')  # convert from Tensor image

In [ ]:
# obtain one batch of training images
dataiter = iter(train_loader)
images, labels = dataiter.__next__()
images = images.numpy() # convert images to numpy for display

# plot the images in the batch, along with the corresponding labels
fig = plt.figure(figsize=(25, 8))
# display 10 images
for idx in np.arange(10):
    ax = fig.add_subplot(2, 5, idx+1, xticks=[], yticks=[])
    imshow(images[idx])
    ax.set_title(classes[labels[idx]])

    # take a look at the shape of one image
    if idx == 0:
        print(images[idx].shape) # grayscale image: 1x128x128

## Define the Network [Architecture](http://pytorch.org/docs/stable/nn.html)

Now let's define our CNN. A network with two convolutional layers is shown in the image below, and you've been given starter code with two convolutional and two maxpooling layers.

<img src='images/2_layer_conv.png' height=50% width=50% />

In [ ]:
# Model definition
class PneumoniaCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv_layers = nn.Sequential(
            # Input size: 1 x IMG_SIZE x IMG_SIZE
            nn.Conv2d(1, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            # Size after pool: 16 x (IMG_SIZE/2) x (IMG_SIZE/2)
            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )
        self.fc_layers = nn.Sequential(
            nn.Linear(32 * (IMG_SIZE // 4) * (IMG_SIZE // 4), 128),
            nn.ReLU(),
            nn.Linear(128, 2)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1)
        x = self.fc_layers(x)
        return x

model = PneumoniaCNN().to(device)
print(model)


Next, we decide on a loss and optimization function that is best suited for this classification task. 

In [ ]:
def loss_normalizer(loss_function, labels):
    """Denominator of mean CrossEntropyLoss for the hard labels used here."""
    if loss_function.weight is None:
        return labels.numel()
    return loss_function.weight[labels].sum().item()


criterion_A = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

## Train the Network

Remember to look at how the training and validation loss decreases over time; if the validation loss ever increases it indicates possible overfitting.

In [ ]:
# Train the original CNN. Average losses over images, including the last batch.
epochs = 5
train_losses, val_losses = [], []

for epoch in range(epochs):
    model.train()
    train_loss_sum, train_weight_sum = 0.0, 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion_A(outputs, labels)
        loss.backward()
        optimizer.step()

        batch_weight = loss_normalizer(criterion_A, labels)
        train_loss_sum += loss.item() * batch_weight
        train_weight_sum += batch_weight

    model.eval()
    val_loss_sum, val_weight_sum, correct, total = 0.0, 0.0, 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion_A(outputs, labels)
            batch_weight = loss_normalizer(criterion_A, labels)
            val_loss_sum += loss.item() * batch_weight
            val_weight_sum += batch_weight
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)

    train_losses.append(train_loss_sum / train_weight_sum)
    val_losses.append(val_loss_sum / val_weight_sum)
    print(f"Epoch {epoch + 1}/{epochs}.. "
          f"Train Loss: {train_losses[-1]:.3f}.. "
          f"Val Loss: {val_losses[-1]:.3f}.. "
          f"Val Acc: {100 * correct / total:.2f}%")

In [ ]:

plt.plot(train_losses, label='Training loss')
plt.plot(val_losses, label='Validation loss')
plt.legend()
plt.show()


## Test the Trained Network

Test your trained model on previously unseen data! 

In [ ]:
@torch.no_grad()
def report_test_results(network, loader, loss_function, class_names):
    network.eval()
    model_device = next(network.parameters()).device
    num_classes = len(class_names)
    class_correct = torch.zeros(num_classes, dtype=torch.long)
    class_total = torch.zeros(num_classes, dtype=torch.long)
    loss_sum, weight_sum = 0.0, 0.0

    for images, labels in loader:
        images, labels = images.to(model_device), labels.to(model_device)
        outputs = network(images)
        loss = loss_function(outputs, labels)
        batch_weight = loss_normalizer(loss_function, labels)
        loss_sum += loss.item() * batch_weight
        weight_sum += batch_weight

        labels_cpu = labels.cpu()
        predictions = outputs.argmax(1).cpu()
        class_total += torch.bincount(labels_cpu, minlength=num_classes)
        class_correct += torch.bincount(
            labels_cpu[predictions == labels_cpu], minlength=num_classes
        )

    recalls = class_correct.float() / class_total
    loss_label = 'Weighted cross-entropy' if loss_function.weight is not None else 'Cross-entropy'
    print(f'Test {loss_label}: {loss_sum / weight_sum:.6f}')
    for name, recall, correct, total in zip(
        class_names, recalls, class_correct, class_total
    ):
        print(f'Test Recall for {name}: {100 * recall.item():.1f}% '
              f'({correct.item()}/{total.item()})')
    accuracy = class_correct.sum().item() / class_total.sum().item()
    print(f'Overall Test Accuracy: {100 * accuracy:.1f}%')
    print(f'Balanced Test Accuracy: {100 * recalls.mean().item():.1f}%')
    return {
        'loss': loss_sum / weight_sum,
        'accuracy': accuracy,
        'balanced_accuracy': recalls.mean().item(),
        'class_recall': recalls.tolist(),
    }


test_metrics_A = report_test_results(model, test_loader, criterion_A, classes)

The overall accuracy looks quite good! But wait a minute...

**TODO:** 1) How do you interpret the results? Are there classes that performed better or worse than others? Why do you think that is? **(2 points)**

**ANSWER:** 

### Visualize Sample Test Results

In [ ]:
# --- Create a shuffled copy of the test loader just for visualization ---
# (We keep the original test_loader with shuffle=False for evaluation)
test_loader_vis = DataLoader(test_data, batch_size=10, shuffle=True)

# Get one random batch of test images
dataiter = iter(test_loader_vis)
images, labels = next(dataiter)

# Move tensors to GPU if available
images = images.to(device)
labels = labels.to(device)

# --- Run the model on this batch ---
model.eval()
with torch.no_grad():
    outputs = model(images)
    _, preds_tensor = torch.max(outputs, 1)

# Move predictions and labels back to CPU for plotting
preds = preds_tensor.cpu().numpy()
labels = labels.cpu().numpy()
images = images.cpu().numpy()

# --- Plot images with predicted and true labels ---
fig = plt.figure(figsize=(20, 6))
for idx in np.arange(len(images)):
    ax = fig.add_subplot(2, 5, idx + 1, xticks=[], yticks=[])
    
    # Unnormalize the image (reverse the normalization)
    img = np.squeeze(images[idx]) * 0.5 + 0.5
    img = np.clip(img, 0, 1)
    ax.imshow(img, cmap='gray')
    
    # Define predicted and true labels
    pred_label = classes[preds[idx]]
    true_label = classes[labels[idx]]
    
    # Color the title: green = correct, red = wrong
    color = "green" if preds[idx] == labels[idx] else "red"
    ax.set_title(f"{pred_label} ({true_label})", color=color)

plt.tight_layout()
plt.show()


Let's see how we can improve it using advanced training techniques.

---
## Improved Training Techniques

Let's start by adding more data augmentation to increase the data variability, and by using a weighted sampler to reduce class imbalance.

**TODO:** 2) Implement at least two more data augmentation steps below! You can find a list of available transforms [here](https://pytorch.org/vision/0.9/transforms.html). You can for example add a `ColorJitter` or a `RandomAffine` transform. Careful: Which transforms do you apply to the training data and which to validation and test data? **(3 points)**

In [ ]:
from collections import Counter
from torch.utils.data import WeightedRandomSampler

# --- Enhanced Data Augmentation ---
# YOUR CODE HERE
train_transforms_improved = 

test_transforms_improved = 

# END STUDENT CODE

# Reload datasets with new transforms
train_data_improved, val_data_improved = load_train_val(
    train_transforms_improved, test_transforms_improved
)
test_data_improved  = datasets.ImageFolder('./data/chest_xray/test', transform=test_transforms_improved)

# --- Balanced sampling; class weights below additionally prioritize NORMAL ---
targets = np.array(train_data_improved.targets)
class_sample_count = np.array([len(np.where(targets == t)[0]) for t in np.unique(targets)])
weights = 1. / class_sample_count
samples_weights = np.array([weights[t] for t in targets])
sampler = WeightedRandomSampler(samples_weights, len(samples_weights))

batch_size = 32
train_loader_improved = DataLoader(train_data_improved, batch_size=batch_size, sampler=sampler)
val_loader_improved   = DataLoader(val_data_improved, batch_size=batch_size, shuffle=False)
test_loader_improved  = DataLoader(test_data_improved, batch_size=batch_size, shuffle=False)

# --- Class Weights for CrossEntropyLoss ---
counts = Counter(train_data_improved.targets)
class_weights = [1.0 / counts[i] for i in range(len(classes))]
class_weights = torch.FloatTensor(class_weights).to(device)

Now we define a model with batch normalization and dropout, and keep a class-weighted
cross-entropy loss to emphasize the less frequent NORMAL class.

Note that sampling and weighting are combined in this solution: the sampler draws
the classes approximately equally often, and the weighted loss further increases the
relative influence of NORMAL examples. This changes the training objective and can trade
PNEUMONIA recall for higher NORMAL recall. Their combination is not required for every dataset.

**TODO:** 3) Define an improved model architecture with batch norm and dropout as regularization
to reduce overfitting. Research these techniques and briefly describe what they do below.
Keep the model small enough for your available hardware. **(6 points)**

In [ ]:
# Modified CNN with more Dropout

class PneumoniaCNN_Improved(nn.Module):
    # YOUR CODE HERE
    
    
    
    # END STUDENT CODE

modelB = PneumoniaCNN_Improved().to(device)
criterion_B = nn.CrossEntropyLoss(weight=class_weights)  # Use class weights to handle imbalance
optimizer = optim.Adam(modelB.parameters(), lr=0.001)

print(modelB)

**ANSWER:** 

Now we implement an improved train loop with early stopping and a learning rate scheduler.

**TODO:** 4) Implement a learning rate scheduler to decrease the learning rate automatically when no improvement is visible anymore. You can find the documentation and an example [here](https://docs.pytorch.org/docs/stable/generated/torch.optim.lr_scheduler.ReduceLROnPlateau.html). **(4 points)**

**HINT:** This includes two steps: 1) Define the scheduler and 2) apply it in the train loop below.

In [ ]:
# YOUR CODE HERE



In [ ]:
from time import perf_counter
import copy

start = perf_counter()
epochs = 5
train_losses_B, val_losses_B = [], []
early_stopping_patience = 4
best_val_loss = float('inf')
epochs_no_improve = 0
best_epoch_B = 0
best_model_wts = copy.deepcopy(modelB.state_dict())

for epoch in range(epochs):
    modelB.train()
    train_loss_sum, train_weight_sum = 0.0, 0.0
    for images, labels in train_loader_improved:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = modelB(images)
        loss = criterion_B(outputs, labels)
        loss.backward()
        optimizer.step()

        # Undo the batch-level division before accumulating the weighted loss.
        batch_weight = loss_normalizer(criterion_B, labels)
        train_loss_sum += loss.item() * batch_weight
        train_weight_sum += batch_weight

    modelB.eval()
    val_loss_sum, val_weight_sum, correct, total = 0.0, 0.0, 0, 0
    with torch.no_grad():
        for images, labels in val_loader_improved:
            images, labels = images.to(device), labels.to(device)
            outputs = modelB(images)
            loss = criterion_B(outputs, labels)
            batch_weight = loss_normalizer(criterion_B, labels)
            val_loss_sum += loss.item() * batch_weight
            val_weight_sum += batch_weight
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)

    train_loss = train_loss_sum / train_weight_sum
    val_loss = val_loss_sum / val_weight_sum
    train_losses_B.append(train_loss)
    val_losses_B.append(val_loss)

    # YOUR CODE HERE
  
  
    # END STUDENT CODE

    print(f"Epoch {epoch + 1}/{epochs}.. Train Loss: {train_loss:.3f}.. "
          f"Val Loss: {val_loss:.3f}.. Val Acc: {100 * correct / total:.2f}% "
          f"(LR: {optimizer.param_groups[0]['lr']:.5f})")

    if val_loss < best_val_loss - 1e-4:
        best_val_loss = val_loss
        best_model_wts = copy.deepcopy(modelB.state_dict())
        best_epoch_B = epoch + 1
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        print(f"No improvement for {epochs_no_improve} epoch(s).")
    if epochs_no_improve >= early_stopping_patience:
        print("Early stopping triggered!")
        break

modelB.load_state_dict(best_model_wts)
modelB.eval()
print(f"Training complete. Best weighted val loss: {best_val_loss:.4f} "
      f"(epoch {best_epoch_B})")
print(f"Runtime training: {perf_counter() - start:.2f} seconds")

### Test

Now let's see what we have gained:

In [ ]:
test_metrics_B = report_test_results(
    modelB, test_loader_improved, criterion_B, classes
)

### Visualize Sample Test Results

In [ ]:
# --- Create a shuffled copy of the test loader just for visualization ---
# (We keep the original test_loader with shuffle=False for evaluation)
test_loader_vis = DataLoader(test_data, batch_size=10, shuffle=True)

# Get one random batch of test images
dataiter = iter(test_loader_vis)
images, labels = next(dataiter)

# Move tensors to GPU if available
images = images.to(device)
labels = labels.to(device)

# --- Run the model on this batch ---
modelB.eval()
with torch.no_grad():
    outputs = modelB(images)
    _, preds_tensor = torch.max(outputs, 1)

# Move predictions and labels back to CPU for plotting
preds = preds_tensor.cpu().numpy()
labels = labels.cpu().numpy()
images = images.cpu().numpy()

# --- Plot images with predicted and true labels ---
fig = plt.figure(figsize=(20, 6))
for idx in np.arange(len(images)):
    ax = fig.add_subplot(2, 5, idx + 1, xticks=[], yticks=[])
    
    # Unnormalize the image (reverse the normalization)
    img = np.squeeze(images[idx]) * 0.5 + 0.5
    img = np.clip(img, 0, 1)
    ax.imshow(img, cmap='gray')
    
    # Define predicted and true labels
    pred_label = classes[preds[idx]]
    true_label = classes[labels[idx]]
    
    # Color the title: green = correct, red = wrong
    color = "green" if preds[idx] == labels[idx] else "red"
    ax.set_title(f"{pred_label} ({true_label})", color=color)

plt.tight_layout()
plt.show()


**TODO:** 5) Plot training and validation losses for the original and improved CNN in
separate panels. Describe their trends. Why does a higher loss for the improved CNN
not automatically imply worse classification performance? Consider the loss weighting,
sampling, augmentation, and dropout. **(4 points)**

In [ ]:
# YOUR CODE HERE



**ANSWER:** 

If you have a powerful hardware available for training, why don't you increase the image resolution and train longer for comparison?

Great job! You have successfully trained an advanced convolutional neural network to detect pneunomia in xray images and handled class imbalance.

---
## Transfer Learning: From ImageNet to Chest X-Rays

So far, we have trained our CNNs from scratch. **Transfer learning** starts from a model
pretrained on a large dataset, such as ImageNet. Its learned features can provide a useful
starting point even for a different task. We replace the original classifier for 1,000
ImageNet classes with a new classifier for our two chest X-ray classes.

- **Feature extraction:** freeze the pretrained backbone and train only the new classifier.
- **Fine-tuning:** additionally update selected pretrained layers with a smaller learning rate.

First, follow the ResNet-18 feature-extraction example. Then implement a fine-tuning variant.
Run the imports and data-loading cells above first. The first model load downloads about
45 MB of pretrained weights; later runs use the local cache. CPU training works, but a GPU
is recommended. The short training runs below are starting points, not optimized results.

References: [PyTorch transfer learning tutorial](https://docs.pytorch.org/tutorials/beginner/transfer_learning_tutorial.html)
and [Torchvision ResNet-18](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.resnet18.html).

### Example: Prepare Inputs for the Pretrained Model

Many pretrained models require **224 × 224 pixel inputs**, **three channels**, and **ImageNet normalization**. We therefore copy the
grayscale image into three channels; this does not add color information, but is needed for the pretrained model. We resize the
whole image rather than center-cropping it. Only training images receive random augmentation.

We create separate loaders for this section and handle class imbalance using a weighted
loss with ordinary shuffled batches. The weights are calculated from the training set only.

In [ ]:
import copy
from torchvision import models
from time import perf_counter

TL_IMG_SIZE = 224
TL_BATCH_SIZE = 16  # Reduce to 8 if GPU memory is limited.
tl_weights = models.ResNet18_Weights.DEFAULT
tl_preprocessing = tl_weights.transforms()

tl_train_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((TL_IMG_SIZE, TL_IMG_SIZE)),
    transforms.RandomRotation(5),
    transforms.ToTensor(),
    transforms.Normalize(mean=tl_preprocessing.mean, std=tl_preprocessing.std),
])
tl_eval_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((TL_IMG_SIZE, TL_IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=tl_preprocessing.mean, std=tl_preprocessing.std),
])

# Reuse the existing split directories, with new preprocessing.
tl_train_data, tl_val_data = load_train_val(
    tl_train_transform, tl_eval_transform
)
tl_test_data = datasets.ImageFolder(test_data.root, transform=tl_eval_transform)
assert tl_train_data.class_to_idx == tl_val_data.class_to_idx == tl_test_data.class_to_idx
tl_classes = tl_train_data.classes

tl_train_loader = DataLoader(tl_train_data, batch_size=TL_BATCH_SIZE, shuffle=True)
tl_val_loader = DataLoader(tl_val_data, batch_size=TL_BATCH_SIZE, shuffle=False)
tl_test_loader = DataLoader(tl_test_data, batch_size=TL_BATCH_SIZE, shuffle=False)

tl_counts = torch.bincount(torch.tensor(tl_train_data.targets), minlength=len(tl_classes))
tl_class_weights = tl_counts.sum() / (len(tl_classes) * tl_counts.float())
tl_criterion = nn.CrossEntropyLoss(weight=tl_class_weights.to(device))
print('Class mapping:', tl_train_data.class_to_idx)
print('Input shape:', tuple(tl_train_data[0][0].shape))

### Example: Replace the Classifier and Freeze the Backbone

Let's look at an example for transfer learning with a pretrained ResNet18 model. First, we load the model with pretrained weights and take a look at the architecture:

In [ ]:
tl_feature_model = models.resnet18(weights=tl_weights) # load pretrained ResNet18 model with default weights
tl_feature_model # print the architecture

From the model output, we can see that the final fully connected layer (`fc`) has 512 input features and 1000 output features (for ImageNet classification). We will replace this layer with a new one that has 2 output features for our binary classification task. We freeze all layers except the final fully connected layer, so we only train the new `fc` layer. `requires_grad = False` prevents parameter updates. The new `nn.Linear` layer has trainable parameters by default. 

In [ ]:
for parameter in tl_feature_model.parameters():
    parameter.requires_grad = False # Freeze all layers (we'll add a new classifier on top with requires_grad=True, so we train only the last layer)

tl_feature_model.fc = nn.Linear(tl_feature_model.fc.in_features, len(tl_classes)) # Replace the final layer with a new one; we need two output neurons for the two classes (Pneumonia, Normal)
tl_feature_model = tl_feature_model.to(device) 
tl_feature_optimizer = optim.Adam(tl_feature_model.fc.parameters(), lr=1e-3) # Optimize only the new classifier

print('Trainable parameters:', sum(p.numel() for p in tl_feature_model.parameters()
                                   if p.requires_grad))
print(tl_feature_model.fc)
print('Overall number of parameters:', sum(p.numel() for p in tl_feature_model.parameters()))

### Train and Validate

The helper below works for both variants and restores the weights with the lowest validation
loss, including the starting weights if no epoch improves on them. We keep BatchNorm running
statistics fixed in both variants: freezing parameters alone does not stop these statistics
from changing. Calling `eval()` on BatchNorm does **not** disable gradients for trainable parameters.

In [ ]:
def tl_run_epoch(network, loader, optimizer=None):
    training = optimizer is not None
    network.train(training)
    if training:
        # Keep pretrained running means/variances fixed, including in layer4.
        for module in network.modules():
            if isinstance(module, nn.BatchNorm2d):
                module.eval()

    loss_sum, weight_sum, correct, total = 0.0, 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        if training:
            optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(training):
            logits = network(images)
            loss = tl_criterion(logits, labels)
            if training:
                loss.backward()
                optimizer.step()

        # Weighted CrossEntropyLoss divides by the sum of target-class weights.
        batch_weight = loss_normalizer(tl_criterion, labels)
        loss_sum += loss.item() * batch_weight
        weight_sum += batch_weight
        correct += (logits.argmax(1) == labels).sum().item()
        total += labels.size(0)
    return loss_sum / weight_sum, correct / total


def tl_fit(network, optimizer, epochs):
    history = {'train_loss': [], 'val_loss': [], 'val_acc': []}
    best_val_loss, _ = tl_run_epoch(network, tl_val_loader)
    best_weights = copy.deepcopy(network.state_dict())
    start = perf_counter()

    for epoch in range(epochs):
        train_loss, _ = tl_run_epoch(network, tl_train_loader, optimizer)
        val_loss, val_acc = tl_run_epoch(network, tl_val_loader)
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        print(f'Epoch {epoch + 1}/{epochs} | Train loss: {train_loss:.3f} | '
              f'Val loss: {val_loss:.3f} | Val accuracy: {100 * val_acc:.1f}%')

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_weights = copy.deepcopy(network.state_dict())

    network.load_state_dict(best_weights)
    network.eval()
    print(f'Best val loss: {best_val_loss:.3f} | Runtime: {perf_counter() - start:.1f} s')
    return history

In [ ]:
TL_FEATURE_EPOCHS = 3
tl_feature_history = tl_fit(tl_feature_model, tl_feature_optimizer, TL_FEATURE_EPOCHS)

### Your Turn: Fine-Tune the Last ResNet Block

Now it's your turn! Sometimes it makes sense to fine-tune more layers, especially if the new dataset is very different from the original dataset. In this case, we can try to fine-tune the last block of ResNet18 (layer4) and the final fully connected layer (fc).

**TODO:** 6) Start from a copy of the trained feature-extraction model and adapt it:

1. Freeze all parameters, then unfreeze `layer4` and `fc`.
2. Create a new Adam optimizer that receives only trainable parameters, using `lr=1e-4`.
3. Train for two additional epochs with `tl_fit`. Keep the feature-extraction model for comparison, and save the new output from `tl_fit` in `tl_finetune_history`.

**(6 points)**



In [ ]:
# start with a copy of the feature model, so we can compare the two approaches (feature extraction vs. fine-tuning)
tl_finetune_model = copy.deepcopy(tl_feature_model).to(device)

# YOUR CODE HERE




### Overall Comparison: All Four Models

Compare the original CNN, improved CNN, frozen ResNet, and fine-tuned ResNet using:

- **Overall accuracy:** the fraction of all examples classified correctly.
- **Class recall:** the fraction of true examples of a class classified correctly.
- **Balanced accuracy:** the mean of the class recalls, giving every class equal weight.

The evaluator below also computes unweighted cross-entropy and a shared class-weighted
cross-entropy on both validation and test data. The common weights are derived only from
training labels. Weighted losses are accumulated over all images before normalization,
including batches that contain a single class. Thus all models use the same evaluation
loss definitions, while retaining their required input preprocessing.

Use validation results for model selection and test results for final reporting. The plots
show each model's own training objective throughout training; their absolute loss levels
reflect different training conditions. The original CNN uses its final epoch, while the
improved CNN and TL models restore their selected validation checkpoints.

In [ ]:
@torch.no_grad()
def evaluate(network, loader, class_names, loss_weights):
    network.eval()
    model_device = next(network.parameters()).device
    num_classes = len(class_names)
    loss_weights = loss_weights.to(model_device)
    correct = torch.zeros(num_classes, dtype=torch.long)
    total = torch.zeros(num_classes, dtype=torch.long)
    ce_sum, weighted_ce_sum, weight_sum = 0.0, 0.0, 0.0

    for images, labels in loader:
        images, labels = images.to(model_device), labels.to(model_device)
        logits = network(images)
        image_losses = nn.functional.cross_entropy(logits, labels, reduction='none')
        image_weights = loss_weights[labels]
        ce_sum += image_losses.sum().item()
        weighted_ce_sum += (image_losses * image_weights).sum().item()
        weight_sum += image_weights.sum().item()

        predictions = logits.argmax(1).cpu()
        labels_cpu = labels.cpu()
        total += torch.bincount(labels_cpu, minlength=num_classes)
        correct += torch.bincount(
            labels_cpu[predictions == labels_cpu], minlength=num_classes
        )

    recalls = correct.float() / total
    num_images = total.sum().item()
    return {
        'loss': ce_sum / num_images,
        'weighted_loss': weighted_ce_sum / weight_sum,
        'accuracy': correct.sum().item() / num_images,
        'balanced_accuracy': recalls.mean().item(),
        'class_recall': recalls.tolist(),
    }


assert classes == tl_classes
assert train_data.class_to_idx == train_data_improved.class_to_idx == tl_train_data.class_to_idx
assert test_data.class_to_idx == test_data_improved.class_to_idx == tl_test_data.class_to_idx

# Use exactly the same class weights for every model's evaluation.
comparison_counts = torch.bincount(
    torch.tensor(train_data.targets), minlength=len(classes)
).float()
comparison_weights = comparison_counts.sum() / (len(classes) * comparison_counts)

experiments = [
    ('CNN (original)', model, val_loader, test_loader,
     {'train_loss': train_losses, 'val_loss': val_losses}, 'Unweighted CE'),
    ('CNN (improved)', modelB, val_loader_improved, test_loader_improved,
     {'train_loss': train_losses_B, 'val_loss': val_losses_B}, 'Class-weighted CE'),
    ('ResNet (frozen backbone)', tl_feature_model, tl_val_loader, tl_test_loader,
     tl_feature_history, 'Class-weighted CE'),
    ('ResNet (fine-tuned)', tl_finetune_model, tl_val_loader, tl_test_loader,
     tl_finetune_history, 'Class-weighted CE'),
]

results, validation_results = {}, {}
for name, network, validation_loader, current_test_loader, history, loss_label in experiments:
    val_result = evaluate(network, validation_loader, classes, comparison_weights)
    result = evaluate(network, current_test_loader, classes, comparison_weights)
    validation_results[name], results[name] = val_result, result

    print(f"\n{name}")
    print(f"  Validation CE: {val_result['loss']:.4f} | "
          f"shared weighted CE: {val_result['weighted_loss']:.4f} | "
          f"accuracy: {100 * val_result['accuracy']:.1f}%")
    print(f"  Test CE: {result['loss']:.4f} | "
          f"shared weighted CE: {result['weighted_loss']:.4f}")
    print(f"  Test accuracy:          {100 * result['accuracy']:.1f}%")
    print(f"  Test balanced accuracy: {100 * result['balanced_accuracy']:.1f}%")
    for class_name, recall in zip(classes, result['class_recall']):
        print(f"  Test recall {class_name}: {100 * recall:.1f}%")

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, (name, network, validation_loader, current_test_loader, history, loss_label) in zip(
    axes.flat, experiments
):
    epoch_numbers = range(1, len(history['train_loss']) + 1)
    ax.plot(epoch_numbers, history['train_loss'], label='Training')
    ax.plot(epoch_numbers, history['val_loss'], label='Validation')
    ax.set(title=name, xlabel='Epoch in this training stage', ylabel=loss_label)
    ax.set_xticks(list(epoch_numbers))
    ax.legend()
plt.tight_layout()
plt.show()

### Final Interpretation: Training, Generalization, and Transfer Learning

**TODO:** 7) Interpret the complete experiment.

a) Compare all four networks using test accuracy, balanced accuracy, and the recall of
   both classes. Which model provides the most balanced performance in your run?
   
b) Explain the training and validation curves, including the effects of augmentation,
   dropout, sampling, and loss weighting. Why can a model have a higher loss but better
   NORMAL recall? Use the shared validation losses for direct model comparisons, and
   distinguish validation performance from test performance.

c) Did fine-tuning improve both classes compared with the frozen backbone? Explain why
   a smaller learning rate is useful for pretrained layers, and discuss what the results
   do and do not establish about the benefit of pretraining.

**(6 points)**

**ANSWER:**

